In [ ]:
# Rest APIs
import requests
import httpx
from urllib.request import urlopen
import time

# disable ssl verification
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

### Check API Status

In [ ]:
# url = "http://127.0.0.1:8001"
url = "https://watchtower.finanssure.com"
response = requests.get(url)

print(f"Status: {response.status_code} | Elapsed: {response.elapsed.total_seconds()*1000} ms")

### Measure Performance of http clients

In [ ]:
%timeit -n 20 urlopen(url).getcode()

In [ ]:
%timeit -n 20 session.get(url).status_code

In [ ]:
%timeit -n 20 requests.get(url).status_code

In [ ]:
%timeit -n 20 httpx.get(url).status_code

In [ ]:
url = 'https://api.github.com'
t0 = time.time()
response = urlopen(url)
t1 = (time.time()-t0) * 1000

print(f"Status: {response.getcode()} | Elapsed: {t1} ms")

### Check Server Status

In [ ]:
import socket
import time

def check_server(host, port, timeout=2):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM) #presumably
    sock.settimeout(timeout)
    t0 = time.time()
    try:
       sock.connect((host,port))
    except: 
        is_success = False
    else: 
        sock.close()
        is_success = True
    
    t1 = (time.time()-t0) * 1000
    return is_success, round(t1, 2)

In [ ]:
# Ping Servers
host = "127.0.0.1"
port = 6379

status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")


In [ ]:
# SSH Servers
host = "127.0.0.1"
port = 22

status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")

### Check Website Status

In [ ]:
# Websites
host = "watchtower.finanssure.com"
port = 443
status_code, elapsed = check_server(host, port)
print(f"Status: {status_code} | Elapsed: {elapsed}")

### Check Domain Expiry

In [ ]:
import whois

def check_domain_expiry(domain_name):
    try:
        domain_info = whois.whois(domain_name)
        print(domain_info)
        expiry_date = domain_info.expiration_date
        print(expiry_date)

        # Handle the possibility of multiple expiration dates
        if isinstance(expiry_date, list):
            expiry_date = expiry_date[0]

        current_date = datetime.utcnow()

        if expiry_date is None:
            print(f"Could not retrieve expiration date for {domain_name}.")
        else:
            remaining_days = (expiry_date - current_date).days
            print(f"Domain {domain_name} expires on {expiry_date}.")
            print(f"Days until expiry: {remaining_days}")

            if remaining_days < 0:
                print("The domain has already expired.")
            elif remaining_days <= 30:
                print("The domain will expire soon.")
            else:
                print("The domain is valid.")

    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage
check_domain_expiry('finanssure.com')

### Check Certificate Expiry

In [ ]:
import ssl
import socket
from datetime import datetime
import certifi

def check_certificate_expiry(hostname, port=443):
    context = ssl.create_default_context(cafile=certifi.where())

    with socket.create_connection((hostname, port)) as sock:
        with context.wrap_socket(sock, server_hostname=hostname) as ssock:
            cert = ssock.getpeercert()

    if not cert:
        print(f"Could not retrieve certificate for {hostname}")
        return

    # Get the certificate's expiration date
    exp_date_str = cert['notAfter']
    exp_date = datetime.strptime(exp_date_str, '%b %d %H:%M:%S %Y %Z')

    # Get the current date
    current_date = datetime.utcnow()

    remaining_days = (exp_date - current_date).days

    print(f"Certificate for {hostname} is valid until {exp_date}, with {remaining_days} days remaining.")

    if remaining_days < 0:
        print(f"The certificate has expired.")
    elif remaining_days <= 30:
        print(f"The certificate will expire soon: {remaining_days} days remaining.")
    else:
        print(f"The certificate is valid.")

# Example usage
hostname = 'finanssure.com'
check_certificate_expiry(hostname)

### Check Database Connection

In [ ]:
# Database
from sqlalchemy import create_engine

def check_connection(conn_str):
    engine = create_engine(conn_str)
    t0 = time.time()
    try:
        conn = engine.connect()
    except: 
        is_success = False
    else: 
        conn.close()
        is_success = True
    
    t1 = (time.time()-t0) * 1000
    return is_success, round(t1, 2)

In [ ]:
engine = create_engine('postgresql://POSTGRES_USER:POSTGRES_PASS@DB_HOST:DB_PORT')


In [ ]:
engine.connect()

In [ ]:
conn_str = 'postgresql://POSTGRES_USER:POSTGRES_PASS@DB_HOST:DB_PORT/'

status_code, elapsed = check_connection(conn_str)
print(f"Status: {status_code} | Elapsed: {elapsed}")

### DNS Check

In [ ]:
import dns.resolver
import dns.exception

def check_dns_server(hostname, dns_server, record_type='A'):
    try:
        # Create a resolver instance
        resolver = dns.resolver.Resolver()

        # Set the DNS server to query
        resolver.nameservers = [dns_server]

        # Perform the DNS query
        answer = resolver.resolve(hostname, record_type)
        # print(answer.__dict__)
        # print(answer.response)
        print(answer.rrset.items)
        print(len(answer.rrset.items))

        # Print the response IP address
        if record_type == 'A':
            for rdata in answer:
                print(f"{hostname} resolves to {rdata.address} using DNS server {dns_server}")

        return answer

    except dns.resolver.NoNameservers:
        print(f"No nameservers are available to resolve {hostname} with DNS server {dns_server}.")

    except dns.resolver.NXDOMAIN:
        print(f"The domain {hostname} does not exist.")

    except dns.resolver.Timeout:
        print(f"Query timed out when resolving {hostname} with DNS server {dns_server}.")

    except dns.exception.DNSException as e:
        print(f"An error occurred: {e}")

# Example usage
hostname = 'finanssure.com'
dns_server = '8.8.8.8'  # Google public DNS server
dns_answer = check_dns_server(hostname, dns_server)
dns_answer.__dict__

### Send Slack Notification

In [ ]:
import os
from dotenv import load_dotenv

base_dir = os.getcwd()
load_dotenv(f"{base_dir}/.env")
load_dotenv(f"{base_dir}/.env.local", override=True)

In [ ]:
# https://watchtower.finanssure.com/oauth/callback/slack/?code=ACCESS_CODE&state=
res = requests.post(
    'https://slack.com/api/oauth.v2.access',
    data={
        'client_id': os.getenv('SLACK_CLIENT_ID'),
        'client_secret': os.getenv('SLACK_CLIENT_SECRET'),
        'code': '<ACCESS_CODE>',
        'redirect_uri': 'https://watchtower.finanssure.com/oauth/callback/slack/'
    }
)
res.json()

In [ ]:
from slack_sdk import WebClient
client = WebClient(token=os.getenv('SLACK_BOT_TOKEN'))

response = client.chat_postMessage(
    channel='CHANNEL_ID',
    text='test message'
)
print(response)

In [ ]:
url = "https://slack.com/api/chat.postMessage"

payload = {
    'channel': 'CHANNEL_ID',
    'text': 'Test Slack Message'
}

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {os.getenv("SLACK_BOT_TOKEN")}'
}

response = requests.post(url, json=payload, headers=headers)
print(response.json())